# 08. Frequencies

- Teaches the calculation that asks how the energy curves around a geometry: move one atom a little in each direction, watch how the forces change, and read off the frequencies at which that atom vibrates
- The first module that needs a second derivative of the energy, and the first whose result can come out imaginary
- Four vibrational analyses on three structures: an O atom relaxed at a three-fold hollow, an OH group relaxed at the same hollow and then relaxed again to a tighter force target, and an O atom placed at a bridge site and not relaxed at all
- Two of those three structures give at least one imaginary frequency, one of them on purpose; the section on an imaginary frequency is where that is read
- The last section runs the same analysis with one terminal command, `mlip freq run`, and checks that it gives the numbers the Python sections gave
- Before you start: modules 00 to 07 finished, the `mlip-mace` environment active, and Jupyter Lab running
    - The kernel shown at the top right of this notebook is "Python 3 (ipykernel)", the Python of that environment
    - The model weights sit in `~/.cache/mace/` from step 10 of module 00, so nothing is downloaded here
- The model is `mace` (MACE-MP-0) in every relaxation and in the one call that attaches it from Python, canon `C1`, and the structures are the Pt(1 1 1) slab of module 04 with the adsorbates of module 05
- After this you can relax a structure, run a vibrational analysis on its adsorbate, read a zero-point energy and a vibrational entropy out of the modes, say which mode is which from the displacements, and tell an imaginary mode from a real one and say what it does and does not prove, and run the same analysis with `mlip freq run` and read the record it writes

### Before you run this for a project

- The values in this notebook are teaching values, chosen so it runs on a laptop in minutes
- In a project, the items below are decided by your supervisor and written in the
  project's `CANON.md`; they are never taken from a tool default
    - `C1` MLIP model and head: this notebook uses `mace` (MACE-MP-0); the project may use another
- If a value is not recorded, ask before running; the tool will not stop you

## 0. The setup cell

- Run this cell first, every time you open the notebook
- This notebook writes files and also attaches a model from Python, so it takes a little from each of the two modules before it
    - The relaxations are `mliprun` commands, as in module 07, and each writes one folder into `work/`
    - The frequencies are computed from Python with a model attached to a structure, as in module 06, and each run writes one small folder of its own into `work/` as well
    - The last section computes them once more with the command `mlip freq run`, which writes one folder per run into `work/`
- `sys.path.insert(0, str(Path("../../scripts").resolve()))` puts the repository's `scripts` folder at the front of the list of folders Python searches, as in modules 05 to 07, so `from onboarding_structures import ...` works
- `sys`, `pathlib`, `io`, `json`, `logging`, `shutil`, `subprocess`, `time`, `warnings` and `contextlib` all come with Python
- `numpy` is imported under the short name `np` and `pandas` under `pd`, as in modules 01 to 07
- `write` and `read` come from `ase.io`, as in modules 05 and 07
- `Vibrations` comes from `ase.vibrations`, and it is the one new class of this notebook
- Two functions come out of `mliprun` here: `setup_calculator`, which builds a model and attaches it to a structure, and `calc_fmax`, which reduces a forces array to the length of its longest row
- `MLIP = Path(sys.executable).parent / "mlip"` is the `mlip` command of this environment, the same line module 07 used
    - In the terminal you type `mlip` on its own, with the `mlip-mace` environment active, and that is what the command lines printed below show
- `run_mlip(arguments)` prints the command as you would type it, runs it, prints how long it took, and stops the notebook if the command itself failed
    - `relax(name, atoms, fmax)` writes a structure into `work/<name>/POSCAR` and runs one relaxation on it, so the three relaxations of this notebook are three calls with the command printed each time
    - `final_stage(folder)` reads the run record of a relaxation and returns its last stage, which is where the outcome sits, as module 07's section on reading the outputs laid out
- `shorten(text)` and `last_lines(text, n)` replace this computer's own folder names in a block of terminal output and keep its last lines; they are used on the error output of a failed command and on the help text of the last section
- `LOAD_LOG = io.StringIO()` is a text buffer in memory, and the cell that builds the model writes the loader's own printed lines into it instead of into the notebook, exactly as module 06's section on attaching a calculator describes

In [1]:
import io
import json
import logging
import shutil
import subprocess
import sys
import time
import warnings
from contextlib import redirect_stdout
from pathlib import Path

import numpy as np
import pandas as pd
from ase.io import read, write
from ase.vibrations import Vibrations

sys.path.insert(0, str(Path("../../scripts").resolve()))
from onboarding_structures import pt111_with_adsorbate

from mliprun.cli.utils import setup_calculator
from mliprun.core.utils import calc_fmax

WORK = Path("work")
WORK.mkdir(exist_ok=True)
LOAD_LOG = io.StringIO()

MLIP = Path(sys.executable).parent / "mlip"
HERE = Path.cwd()
ENVIRONMENT = Path(sys.executable).parent.parent
HOME = Path.home()


def shorten(text):
    """Replace this computer's own folder names in a block of terminal output."""
    return (text.replace(str(HERE) + "/", "")
                .replace(str(ENVIRONMENT), "<environment>")
                .replace(str(HOME), "~"))


def last_lines(text, n):
    """The last n lines of a block of output that carry anything."""
    kept = [line for line in shorten(text).splitlines() if line.strip()]
    return "\n".join(kept[-n:])


def run_mlip(arguments):
    """Run one mlip command and return what subprocess.run gives back."""
    print("Command, as you would type it in the terminal")
    print(" ".join(["mlip"] + arguments))
    started = time.perf_counter()
    result = subprocess.run([str(MLIP)] + arguments, capture_output=True, text=True)
    print()
    print("Seconds the command took")
    print(round(time.perf_counter() - started, 1))
    if result.returncode != 0:
        print()
        print("The command failed, and this is the end of what it wrote to its error stream")
        print(last_lines(result.stderr, 15))
        raise SystemExit("mlip command failed")
    return result


def final_stage(folder):
    """The last stage of the run record mliprun wrote in that folder."""
    record = json.loads((Path(folder) / "mliprun_run.json").read_text())
    return record["stages"][-1]


def relax(name, atoms, fmax):
    """Write one structure into work/<name>/POSCAR and relax it there."""
    folder = WORK / name
    folder.mkdir(parents=True, exist_ok=True)
    write(folder / "POSCAR", atoms, format="vasp", direct=True, sort=True)
    run_mlip(["optimize", "run",
              "--structure", str(folder / "POSCAR"),
              "--mlip", "mace",
              "--device", "cpu",
              "--fmax", str(fmax),
              "--optimizer", "bfgs",
              "--max-steps", "200"])
    return folder


print("Folder this notebook writes into")
print(WORK)
print("The mlip command of this environment was found")
print(MLIP.exists())

Folder this notebook writes into
work
The mlip command of this environment was found
True


## 1. Why frequencies

- A relaxation asked which way the energy falls; a frequency calculation asks how it curves
    - The forces are the first derivatives of the energy with respect to the positions of the atoms, as module 06's section on energy and forces said
    - The curvatures are the second derivatives, one number for every pair of coordinates, and that table of numbers is called the Hessian
    - This notebook builds it by finite differences: move one atom a little, ask the model for the forces again, and divide the change in the force by the size of the move
    - The frequencies come out of that table once each coordinate has been weighted by the mass of its atom
- The harmonic approximation is what makes a frequency possible at all
    - It replaces the real energy surface around the geometry by a parabola along every direction
    - It is good for small displacements about a minimum and gets worse the further the atoms move; it says nothing at all about a bond breaking
    - Every number in this notebook is a harmonic number, and the word is not decoration: an anharmonic treatment is a different calculation
- Zero-point energy is the first thing frequencies buy you
    - A harmonic oscillator is never at rest: its lowest state sits half a quantum of energy above the bottom of the parabola
    - Summed over the modes that is `ZPE = Σ ħω/2`, an energy every structure carries and that no relaxation reports
    - It does not cancel between two structures whose bonds differ, so an adsorption energy with it and without it are two different numbers
- Vibrational entropy is the second
    - The same modes are where an adsorbate's entropy comes from at a finite temperature, and a free energy needs it: `ΔG = ΔE + ΔZPE − TΔS`
    - The section on zero-point energy prints both of those terms for one structure
- An imaginary frequency is the third, and it is a diagnostic rather than a quantity
    - A curvature can come out negative, which means the energy falls away along that direction instead of rising; the square root of a negative number is imaginary, and that is how the tool reports it
    - A minimum has no imaginary mode at all
    - A first-order saddle, the kind a transition state sits on, has exactly one, along the direction the reaction runs
    - So counting the imaginary modes is how you find out what kind of point a converged relaxation actually stopped at, and two of the structures below turn out not to be minima

## 2. Relax first

- A harmonic frequency is only meaningful at a stationary point, a geometry where the forces have already gone to nearly zero
    - At a stationary point the energy has no slope left, so the first thing a small displacement changes is the curvature, and a curvature is what a frequency is made of
    - Away from one the energy still has a slope; the atoms do not oscillate about such a geometry at all, and the numbers the same calculation prints are curvatures of a point nothing sits at
    - The section on an imaginary frequency runs exactly that case on purpose, and the forces printed there are what "not a stationary point" looks like in numbers
- So the structure is relaxed first, with the same command and the same values as module 07
    - `mlip optimize run --structure work/O_fcc/POSCAR --mlip mace --device cpu --fmax 0.05 --optimizer bfgs --max-steps 200`
    - 0.05 eV/Å is module 07's teaching value for the force target, canon `C2` there, and `bfgs` with a ceiling of 200 steps are its other two values
    - Module 07 also said that a frequency analysis is usually run tighter than an energy; the section on the same OH relaxed tighter is where this notebook tests that claim against its own numbers
- `read` brings the relaxed geometry back, and the constraint comes back with it
    - `opt_final.vasp` carries `F F F` on the nine atoms of the bottom layer, and ASE collects every atom whose three flags are all `F` into one `FixAtoms` object holding those nine indices
    - That matters in the next section: left to itself, `Vibrations` vibrates every atom that no `FixAtoms` holds
    - `sort=True` when the file was written grouped the atoms by species, so the single O is the first atom of this file; the cell finds it by its symbol rather than by that position, as module 05's section on saving the structures said to
- `np.sort(array)` puts the values of an array in increasing order, used here on the distances from the O to every Pt atom, so the first three are its three nearest neighbours
- In the printed output:
    - The run record's own three numbers, the ones canon `K1` is about: the steps taken, the `converged` label and the force the run reached
    - The energy the run ended on, −158.819 eV, which is a total energy of this model and says nothing on its own, as module 06's section on energy and forces put it
    - Module 07 relaxed this same structure with this same command in its section on running one relaxation from the command line, and reported the same 16 steps, the same 0.0442 eV/Å and the same −158.819 eV in its section on reading the outputs
    - One `FixAtoms` constraint holding 9 of the 28 atoms
    - The three nearest Pt atoms at the same distance to three decimals, which is module 05's check that the O is still in a three-fold hollow after the relaxation, and the same 2.113 Å module 07 measured on this structure

In [2]:
o_folder = relax("O_fcc", pt111_with_adsorbate("O", "fcc"), 0.05)
o_stage = final_stage(o_folder)

print()
print("Steps the relaxation took")
print(o_stage["steps"])
print("converged, from the last stage's results")
print(o_stage["results"]["converged"])
print("Largest force it reached (eV/Å)")
print(round(o_stage["results"]["final_fmax_eV_per_A"], 4))
print("Final energy (eV)")
print(round(o_stage["results"]["final_energy_eV"], 3))

relaxed = read(o_folder / "opt_final.vasp")
symbols = relaxed.get_chemical_symbols()
i_o = symbols.index("O")
pt_indices = [i for i, symbol in enumerate(symbols) if symbol == "Pt"]
distances = np.sort(relaxed.get_distances(i_o, pt_indices, mic=True))

print()
print("Atoms in the relaxed structure read back")
print(len(relaxed))
print("Constraints it carries")
print(relaxed.constraints)
print("Atoms held by that constraint")
print(len(relaxed.constraints[0].get_indices()))

print()
print("Index of the O in this file")
print(i_o)
print("Distances from the O to its three nearest Pt atoms (Å)")
print([round(float(value), 3) for value in distances[:3]])

Command, as you would type it in the terminal
mlip optimize run --structure work/O_fcc/POSCAR --mlip mace --device cpu --fmax 0.05 --optimizer bfgs --max-steps 200



Seconds the command took
10.7

Steps the relaxation took
16
converged, from the last stage's results
True
Largest force it reached (eV/Å)
0.0442
Final energy (eV)
-158.819

Atoms in the relaxed structure read back
28
Constraints it carries
[FixAtoms(indices=[13, 18, 19, 20, 21, 22, 23, 24, 25])]
Atoms held by that constraint
9

Index of the O in this file
0
Distances from the O to its three nearest Pt atoms (Å)
[2.113, 2.113, 2.113]


## 3. Vibrations of the adsorbate

- `Vibrations(atoms, indices=[i_o], name="work/vib_O")` from `ase.vibrations` sets the calculation up, and `run()` does it
    - `indices` is the list of atoms to vibrate; one atom has three directions, so one atom gives three modes
    - Leave `indices` out and ASE vibrates every atom that no `FixAtoms` holds, which is 19 atoms here: 57 modes and 115 force calls instead of 3 and 7
    - `name` is a folder, not a file: `run()` writes one small `.json` file per displacement into it, and a second `run()` reads those files instead of recomputing, so an interrupted run carries on where it stopped
    - `delta` is the size of each displacement and defaults to 0.01 Å; `nfree` defaults to 2, meaning each direction is displaced both ways and the change in the force is taken between the two
    - It reads `atoms.calc` when it is created, so the model has to be attached before that line and not after it
- What `run()` costs here: seven force calls from `Vibrations` to the calculator, one for the geometry as it stands and two along each of three directions, which is one call per file the cell reports, and the seconds the cell prints cover all seven
- `Vibrations` asks the calculator for the forces rather than asking the structure for them, so the difference module 06's section on energy and forces pointed out — that `atoms.get_forces()` writes exact zeros into the rows of the atoms held by `FixAtoms` — does not apply inside this Hessian; the constraint reaches this calculation only through the default `indices` above
- The model is attached with `setup_calculator(atoms, mlip="mace", device="cpu")`, module 06's one-call attach, and the lines around that call hold back what the loader prints for the length of the call and no longer
    - `with redirect_stdout(LOAD_LOG):` sends the loader's printed lines into the buffer of the setup cell, one of them a weights path that is different on every computer
    - `with warnings.catch_warnings():` and `warnings.simplefilter("ignore")` hide the warnings raised inside that block and nowhere else
    - The `WARNING:root:Default dtype float32 ...` line arrives through Python's logging system instead, which that filter does not reach, so the root logger is turned down for the load and put back immediately afterwards
    - `try:` with `finally:` is what makes putting it back certain: even if the load fails and stops the cell, the `finally` block still runs, so a failed load cannot leave the logger turned down for the rest of the session
    - Module 06's section on attaching a calculator is where all of this is spelled out, and what is being held back is listed there too
- The same calculator object is then used for every structure of this notebook, which is module 06's section on reusing one calculator: the model is built once, and building it is the expensive part
- `get_frequencies()` returns one complex number per mode, in cm⁻¹
    - A real mode comes back with a zero imaginary part and an imaginary mode comes back with a zero real part
    - So an imaginary mode is found by testing the imaginary part of the number, never by looking for a minus sign in front of it
- `summary(log=buffer)` writes the same list as a small table into whatever object it is given, so an `io.StringIO()` buffer catches it and the cell prints it as one block
- `get_mode(n)` is the displacement pattern of mode `n`, one row per atom of the whole structure, with zeros on every atom that was not vibrated
    - It is a normalized eigenvector divided by the square root of each mass, so the overall size of a row carries no physical scale
    - What can be read off it is the direction of the motion and the ratio between two atoms, and that is what the last columns of the table below are
    - The `moves` column of an atom is the length of that atom's row, and the columns of two atoms can be compared with each other because both are displacements of one mode in the same units, each already divided by the square root of its own atom's mass
    - The `along z` column is the share of that atom's motion that lies along the surface normal: 0 for a motion entirely in the surface plane and 1 for one entirely along the normal
- NumPy and pandas in the cell below
    - `np.linalg.norm(vector)` is the length of a vector, used on one atom's row of a mode to get how far that atom moves
    - `frequencies.imag` and `frequencies.real` are the imaginary and the real part of every entry of a complex array
    - `(frequencies.imag != 0).sum()` counts the entries whose imaginary part is not zero, which is the number of imaginary modes, and `int(...)` makes it a plain number
    - `pd.DataFrame(rows).to_string(index=False)` prints a list of dictionaries as an aligned table without row numbers, as in modules 01 to 07
- In the printed output:
    - The largest force this notebook's own model finds on the relaxed structure, next to the one the relaxation reported, and the two agree to the four decimals printed, which is the check that the geometry came back from the file as the relaxation left it
    - Three modes, none of them imaginary, so this relaxed geometry is a minimum of this model as far as the O atom can tell
    - Two of the three come out equal to better than 0.01 cm⁻¹ out of 311 and both move the O in the surface plane; the third is higher and moves it along the surface normal
    - The two that pair up are the two in-plane directions, which the three-fold symmetry of a hollow site makes equivalent, so the exact answer is one frequency twice; the difference printed under the table is how far the finite differences of a single-precision model fall short of that, and it differs from run to run
    - The three equal Pt distances of the previous section are the same symmetry measured on the geometry instead of on the modes
    - The `O moves` column is 0.25 in all three rows, and that is what a normalized eigenvector over a single atom has to give: one divided by the square root of the mass of the O in amu, the same number whichever way it moves
    - `summary()` gives each mode in meV as well as in cm⁻¹ and rounds to one decimal, so the two in-plane modes read as one number there

In [3]:
root_logger = logging.getLogger()
previous_level = root_logger.level

started = time.perf_counter()
try:
    with warnings.catch_warnings(), redirect_stdout(LOAD_LOG):
        warnings.simplefilter("ignore")
        root_logger.setLevel(logging.ERROR)
        setup_calculator(relaxed, mlip="mace", device="cpu")
finally:
    root_logger.setLevel(previous_level)
load_seconds = time.perf_counter() - started
calc = relaxed.calc

print("Calculator now attached to the relaxed structure")
print(type(calc).__name__)
print("Seconds the model took to build")
print(round(load_seconds, 1))
print("Lines the loader printed, held in the buffer instead of the notebook")
print(len(LOAD_LOG.getvalue().strip().splitlines()))

print()
print("Largest force on any atom, from the model in this notebook (eV/Å)")
print(round(calc_fmax(relaxed.get_forces()), 4))
print("Largest force the relaxation itself reported (eV/Å)")
print(round(o_stage["results"]["final_fmax_eV_per_A"], 4))


def mode_rows(vib, atoms, indices):
    """One row per mode: its frequency, whether it is imaginary, and where it moves."""
    names = atoms.get_chemical_symbols()
    rows = []
    for n, value in enumerate(vib.get_frequencies()):
        imaginary = bool(value.imag != 0)
        row = {"mode": n,
               "frequency (cm⁻¹)": round(float(value.imag if imaginary else value.real), 1),
               "imaginary": imaginary}
        pattern = vib.get_mode(n)
        for i in indices:
            length = float(np.linalg.norm(pattern[i]))
            row[f"{names[i]} moves"] = round(length, 4)
            row[f"{names[i]} along z"] = round(float(abs(pattern[i][2]) / length), 3)
        rows.append(row)
    return rows


started = time.perf_counter()
vib_o = Vibrations(relaxed, indices=[i_o], name="work/vib_O")
vib_o.run()
vib_seconds = time.perf_counter() - started
frequencies = vib_o.get_frequencies()

print()
print("Seconds the seven force calls took")
print(round(vib_seconds, 1))
print("Files the run wrote into work/vib_O")
print(vib_o.cache.filecount())
print("Modes")
print(len(frequencies))
print("Imaginary modes among them")
print(int((frequencies.imag != 0).sum()))

print()
print("The three modes of the O atom at the relaxed fcc site")
print(pd.DataFrame(mode_rows(vib_o, relaxed, [i_o])).to_string(index=False))

print()
print("Difference between the two in-plane frequencies (cm⁻¹)")
print(round(float(abs(frequencies[1].real - frequencies[0].real)), 4))

buffer = io.StringIO()
vib_o.summary(log=buffer)
print()
print("What summary() writes")
print(buffer.getvalue())

Calculator now attached to the relaxed structure
MACECalculator
Seconds the model took to build
3.0
Lines the loader printed, held in the buffer instead of the notebook
3

Largest force on any atom, from the model in this notebook (eV/Å)


0.0442
Largest force the relaxation itself reported (eV/Å)
0.0442



Seconds the seven force calls took
1.5
Files the run wrote into work/vib_O
7
Modes
3
Imaginary modes among them
0

The three modes of the O atom at the relaxed fcc site
 mode  frequency (cm⁻¹)  imaginary  O moves  O along z
    0             311.3      False     0.25        0.0
    1             311.3      False     0.25        0.0
    2             419.6      False     0.25        1.0

Difference between the two in-plane frequencies (cm⁻¹)
0.0012

What summary() writes
---------------------
  #    meV     cm^-1
---------------------
  0   38.6     311.3
  1   38.6     311.3
  2   52.0     419.6
---------------------
Zero-point energy: 0.065 eV



## 4. Zero-point energy

- `get_zero_point_energy()` adds up half of each mode's energy, `ZPE = Σ ħω/2`, and returns the sum in eV
    - It is half the sum of the real parts of the mode energies, so a mode whose energy is imaginary contributes nothing to it and nothing is printed to tell you that happened
    - The cell prints the two numbers side by side for this structure, where every mode is real and the two therefore agree; the section on what a zero-point energy does with an imaginary mode is where they part company in meaning
    - `energies.real.sum()` is the sum of the real parts of a complex array, which is that check written out
- What a zero-point energy is for: it is a term of a free energy, `ΔG = ΔE + ΔZPE − TΔS`
    - `ΔE` is the energy difference modules 06 and 07 assembled, `ΔZPE` is the same difference taken over zero-point energies, and `ΔS` is the entropy difference
    - Each term of `ΔZPE` needs its own frequency calculation, on its own relaxed structure, with the same model and the same choice of which atoms are vibrated on both sides
    - The zero-point energy printed below covers the three modes of one O atom and nothing else, so it is the adsorbate's contribution and not the structure's: the Pt atoms have modes too, and they are not in this Hessian
- `HarmonicThermo(vib_energies=energies)` from `ase.thermochemistry` is where the entropy term comes from
    - It takes the list of mode energies in eV, the one `get_energies()` returns, and treats every one of them as a harmonic oscillator
    - `get_entropy(300.0, verbose=False)` returns the vibrational entropy at 300 K in eV/K, and `verbose=False` keeps the class's own breakdown table out of the output
    - Multiplied by the temperature it becomes the `TΔS` term in eV, which is the number to compare a zero-point energy against
- In the printed output:
    - A zero-point energy of 0.0646 eV for the three modes, and half the sum of the real mode energies as the same number
    - `T` times the entropy of those same three modes at 300 K is 0.0473 eV, of the same size as the zero-point energy, and it enters `ΔG` with the opposite sign
    - Both are small next to the 0.37 to 0.56 eV of relaxation energy module 07 measured, and neither is small next to the 0.125 eV that separated the two hollow sites there, which is the size of gap this course keeps calling a hypothesis rather than a result
    - What enters a free energy is the difference between two such terms rather than either one, and a difference between two similar structures is smaller again
    - `clean()` removes the seven files of the displacement folder and returns how many it removed, so a second run of this notebook recomputes them instead of reading a cache left by an older version of the structure

In [4]:
from ase.thermochemistry import HarmonicThermo

energies = vib_o.get_energies()
zero_point = vib_o.get_zero_point_energy()

print("Zero-point energy of the three modes (eV)")
print(round(zero_point, 4))
print("Half the sum of the real parts of the mode energies (eV)")
print(round(0.5 * float(energies.real.sum()), 4))

thermo = HarmonicThermo(vib_energies=energies)
entropy = thermo.get_entropy(300.0, verbose=False)

print()
print("Vibrational entropy of the same three modes at 300 K (eV/K)")
print(round(float(entropy), 6))
print("T times that entropy at 300 K (eV)")
print(round(300.0 * float(entropy), 4))

print()
print("Displacement files removed by clean()")
print(vib_o.clean())

Zero-point energy of the three modes (eV)
0.0646
Half the sum of the real parts of the mode energies (eV)
0.0646

Vibrational entropy of the same three modes at 300 K (eV/K)
0.000158
T times that entropy at 300 K (eV)
0.0473

Displacement files removed by clean()
7


## 5. OH on the surface

- The same three steps on `pt111_with_adsorbate("OH", "fcc")`: relax it, read the relaxed file back, vibrate the adsorbate
    - Two atoms are vibrated this time, the O and the H, so there are six modes and thirteen force calls, one per file the cell reports
    - `sort=True` grouped the atoms of the file by species and H comes before O in that grouping, so here the H is the first atom of the file and the O is the second; both are found by their symbol
    - The calculator is the object built in the section on vibrations of the adsorbate, attached with `atoms.calc = calc` instead of built again
- `indices=sorted([i_o_oh, i_h_oh])` hands the two atoms over in increasing order, and the `sorted` is not optional
    - In ASE 3.29.0, `Vibrations` builds the Hessian in the order the list gives, but turns it into frequencies with the masses taken in increasing atom order, and places the mode patterns back in that order too (`ase/vibrations/data.py`, the lines that read `self._atoms[self.get_mask()]`)
    - Here the O is atom 1 and the H atom 0, so the list `[i_o_oh, i_h_oh]` as it stands gives the O's rows the mass of the H and the H's rows the mass of the O; nothing fails and nothing warns, and every frequency and every mode pattern comes out wrong
    - An earlier version of this notebook passed the list unsorted and printed 90.4i, 1080.5 and 3841.7 cm⁻¹ for this structure, and described its modes from those wrong patterns; the numbers below are the right ones, and the section on `mlip freq run` checks them against a command that sorts the list itself
    - A list of one atom cannot be out of order, so the sections on the O atom alone are not affected
- Which mode is which, read off the table and the line under it rather than guessed
    - The two lowest move both atoms in the surface plane and the same way, the H about eight times as far as the O, so the O–H axis tilts away from the surface normal about a point just below the O, with the H swinging sideways
    - The next one moves both atoms along the surface normal, in the same direction and almost equally far, which is the group moving as a unit against the surface, the adsorbate-surface stretch
    - The pair at 338.2 cm⁻¹ moves both atoms in the surface plane but in opposite directions, the H about twice as far as the O, so the O–H axis tilts about a point between the two atoms while the O slides along the surface
    - The highest moves the H along the O–H axis and against the O, which barely moves, and that axis is the surface normal here, so it is the O–H stretch; it is more than ten times higher than the next mode, which is what a stretch between the lightest atom and a heavy one looks like
    - `np.dot(a, b)` multiplies two vectors entry by entry and adds the results; divided by the length of each it is the cosine of the angle between them, which is 1 when two atoms move the same way and −1 when they move against each other
- Two of the six come out imaginary, and that was not the plan of this section
    - They are the pair at 288.9i cm⁻¹, the tilt with the H swinging sideways, and the relaxation that produced this geometry converged: its `converged` label is true and it reached 0.0452 eV/Å
    - So this model puts an upright OH on the fcc site at a point where tilting it lowers the energy, and a converged relaxation did not find that out
    - A likely reason, which this notebook does not test: the OH was placed exactly upright over the centre of the hollow, where the force along any tilt is zero by symmetry, and a relaxation only follows forces, so it cannot leave a symmetric geometry that a tilt would lower
    - The next section relaxes the same structure to a tighter target, to find out whether the force target is what put them there
    - The section on an imaginary frequency is what an imaginary mode means, and everything it says applies here as much as to the structure that was built to show it
- So the zero-point energy printed for this structure has the same hole in it that the section on what a zero-point energy does with an imaginary mode describes: the two imaginary modes contribute nothing to it
- In the printed output:
    - The relaxation's own three numbers first, then the O–H distance it ended with and the largest force this notebook's model finds on that geometry
    - Then the six modes, with `imaginary` true on the first two rows
    - Then the direction of the H's motion against the O's, one number per mode: the first three move the two atoms the same way and the last three move them against each other
    - Read that line together with the two `moves` columns, because two atoms moving the same way by very different amounts is a tilt and not a translation: the pair at 288.9i moves the O 0.1141 against the H's 0.8863, while the mode at 223.0 moves them 0.2425 against 0.2419
    - Then the zero-point energy, which is not a number to use here, and the count of files `clean()` removed


In [5]:
oh_folder = relax("OH_fcc", pt111_with_adsorbate("OH", "fcc"), 0.05)
oh_stage = final_stage(oh_folder)

print()
print("Steps the relaxation took")
print(oh_stage["steps"])
print("converged, from the last stage's results")
print(oh_stage["results"]["converged"])
print("Largest force it reached (eV/Å)")
print(round(oh_stage["results"]["final_fmax_eV_per_A"], 4))

oh = read(oh_folder / "opt_final.vasp")
oh_symbols = oh.get_chemical_symbols()
i_o_oh = oh_symbols.index("O")
i_h_oh = oh_symbols.index("H")
oh.calc = calc

print()
print("Index of the O and of the H in this file")
print([i_o_oh, i_h_oh])
print("O to H distance after the relaxation (Å)")
print(round(float(oh.get_distance(i_o_oh, i_h_oh, mic=True)), 4))
print("Largest force on any atom, from the model in this notebook (eV/Å)")
print(round(calc_fmax(oh.get_forces()), 4))

vib_oh = Vibrations(oh, indices=sorted([i_o_oh, i_h_oh]), name="work/vib_OH")
vib_oh.run()
frequencies_oh = vib_oh.get_frequencies()

print()
print("Files the run wrote into work/vib_OH")
print(vib_oh.cache.filecount())
print("Modes")
print(len(frequencies_oh))
print("Imaginary modes among them")
print(int((frequencies_oh.imag != 0).sum()))

print()
print("The six modes of the OH group at the relaxed fcc site")
print(pd.DataFrame(mode_rows(vib_oh, oh, [i_o_oh, i_h_oh])).to_string(index=False))

directions = []
for n in range(len(frequencies_oh)):
    pattern = vib_oh.get_mode(n)
    o_step = pattern[i_o_oh]
    h_step = pattern[i_h_oh]
    directions.append(round(float(np.dot(o_step, h_step)
                                  / (np.linalg.norm(o_step) * np.linalg.norm(h_step))), 2))

print()
print("Direction of the H's motion against the O's, per mode, 1 together and -1 opposed")
print(directions)

print()
print("Zero-point energy of these six modes (eV)")
print(round(vib_oh.get_zero_point_energy(), 4))
print("Displacement files removed by clean()")
print(vib_oh.clean())

Command, as you would type it in the terminal
mlip optimize run --structure work/OH_fcc/POSCAR --mlip mace --device cpu --fmax 0.05 --optimizer bfgs --max-steps 200



Seconds the command took
12.6

Steps the relaxation took
22
converged, from the last stage's results
True
Largest force it reached (eV/Å)
0.0452

Index of the O and of the H in this file
[1, 0]
O to H distance after the relaxation (Å)
0.9788
Largest force on any atom, from the model in this notebook (eV/Å)


0.0452



Files the run wrote into work/vib_OH
13
Modes
6
Imaginary modes among them
2

The six modes of the OH group at the relaxed fcc site
 mode  frequency (cm⁻¹)  imaginary  O moves  O along z  H moves  H along z
    0             288.9       True   0.1141        0.0   0.8863        0.0
    1             288.9       True   0.1141        0.0   0.8863        0.0
    2             223.0      False   0.2425        1.0   0.2419        1.0
    3             338.2      False   0.2225        0.0   0.4544        0.0
    4             338.2      False   0.2225        0.0   0.4544        0.0
    5            3758.7      False   0.0607        1.0   0.9662        1.0

Direction of the H's motion against the O's, per mode, 1 together and -1 opposed
[1.0, 1.0, 1.0, -1.0, -1.0, -1.0]

Zero-point energy of these six modes (eV)
0.2888
Displacement files removed by clean()
13


### 5a. The same OH relaxed tighter

- The same structure relaxed to `--fmax 0.01` into `work/OH_fcc_tight/`, with nothing else changed, which is module 07's section on two convergence targets applied to the question the previous section left open
    - Nothing is copied: `pt111_with_adsorbate("OH", "fcc")` builds the same structure again into a second folder, because a second run in the same folder would write over the first
    - Tightening the target is the first thing to try when a small imaginary frequency appears, and this section is that try
- The two imaginary modes survive it
    - The target is five times tighter, the run reaches 0.0092 eV/Å where the first one stopped at 0.0452 eV/Å, and it takes 5 more steps to get there
    - The pair moves from 288.9i to 288.2i cm⁻¹, so the force target is not what put it there, and this notebook does not report it as a convergence artifact
    - The other frequency the table prints, the O–H stretch, shifts by about 11 cm⁻¹ out of 3759 between the two targets, which is the scale on which these numbers are reproducible at all
- What this notebook does not do, and therefore does not claim
    - It does not tilt the OH along that imaginary mode and relax again, which is the check that would say where it goes when it is let loose and whether the symmetry reason of the previous section is the right one, so nothing here says where it ends up
    - It does not enlarge the Hessian beyond the two adsorbate atoms, so it does not say what these six modes would come out at if the Pt atoms under them were vibrated too; which atoms were vibrated is part of the result and is reported with it
    - So what stands is the measurement: at a geometry relaxed to 0.0092 eV/Å, two of the six modes of this OH group are imaginary with this model
- In the table:
    - One row per target, with the steps, the `converged` label, the force reached, the number of imaginary modes, the lowest frequency and the O–H stretch
    - `imaginary` is carried next to that lowest frequency, because 288.2 and 288.9 look like ordinary numbers without it

In [6]:
tight_folder = relax("OH_fcc_tight", pt111_with_adsorbate("OH", "fcc"), 0.01)
tight_stage = final_stage(tight_folder)

tight = read(tight_folder / "opt_final.vasp")
tight_symbols = tight.get_chemical_symbols()
i_o_tight = tight_symbols.index("O")
i_h_tight = tight_symbols.index("H")
tight.calc = calc

vib_tight = Vibrations(tight, indices=sorted([i_o_tight, i_h_tight]), name="work/vib_OH_tight")
vib_tight.run()
frequencies_tight = vib_tight.get_frequencies()

rows = []
for target, stage_here, values in ((0.05, oh_stage, frequencies_oh),
                                   (0.01, tight_stage, frequencies_tight)):
    lowest = values[0]
    rows.append({"fmax asked for (eV/Å)": target,
                 "steps": stage_here["steps"],
                 "converged": stage_here["results"]["converged"],
                 "fmax reached (eV/Å)": round(stage_here["results"]["final_fmax_eV_per_A"], 4),
                 "imaginary modes": int((values.imag != 0).sum()),
                 "lowest mode (cm⁻¹)": round(float(abs(lowest.imag) if lowest.imag != 0
                                                    else lowest.real), 1),
                 "imaginary": bool(lowest.imag != 0),
                 "O–H stretch (cm⁻¹)": round(float(values[-1].real), 1)})

print()
print("The same OH relaxed to two force targets, then vibrated the same way")
print(pd.DataFrame(rows).to_string(index=False))

print()
print("Zero-point energy at the tighter target (eV)")
print(round(vib_tight.get_zero_point_energy(), 4))
print("Displacement files removed by clean()")
print(vib_tight.clean())

Command, as you would type it in the terminal
mlip optimize run --structure work/OH_fcc_tight/POSCAR --mlip mace --device cpu --fmax 0.01 --optimizer bfgs --max-steps 200



Seconds the command took
14.6



The same OH relaxed to two force targets, then vibrated the same way
 fmax asked for (eV/Å)  steps  converged  fmax reached (eV/Å)  imaginary modes  lowest mode (cm⁻¹)  imaginary  O–H stretch (cm⁻¹)
                  0.05     22       True               0.0452                2               288.9       True              3758.7
                  0.01     27       True               0.0092                2               288.2       True              3747.8

Zero-point energy at the tighter target (eV)
0.2881
Displacement files removed by clean()
13


## 6. An imaginary frequency

- The structure here is `pt111_with_adsorbate("O", "bridge")` exactly as the helper places it, with no relaxation at all
    - The bridge site is the point halfway between two Pt atoms of the top layer, and it lies between two of the three-fold hollows
    - Module 07 relaxed this same structure and the O left the bridge site: `bfgs` spent its 200 steps without converging, and `fire` converged it into a hollow with the O 0.735 Å from where it had been placed
    - So it is the placed geometry that is used here, and relaxing it first would destroy the thing this section is about
- What this geometry is, before any frequency is read off it
    - The largest force on any atom is 1.5143 eV/Å, thirty times the target module 07 relaxes to, and the atom carrying it is the O itself
    - Almost all of that force is along the surface normal, 1.514 eV/Å against 0.0318 eV/Å in the surface plane, so the O is mostly at the wrong height rather than in the wrong place sideways
    - A force that size means this is not a stationary point, and the section on relaxing first said what that costs: the three numbers below are curvatures at a point nothing sits at, and not the vibrational frequencies of anything
- What the frequencies do say
    - One of the three comes out imaginary, at 236.0i cm⁻¹, and it moves the O in the surface plane
    - One imaginary mode means the energy falls away along that one direction, so this geometry is not a minimum, and that is the whole of what it means here
- What it does not say, and this is the part to carry out of the module
    - It does not make this a transition state: a transition state is a stationary point with exactly one imaginary mode, and the forces above show that this geometry is not a stationary point
    - So there is no barrier to quote from it and no rate: a barrier is the energy of a saddle measured against a minimum, and nothing here has found a saddle
    - Finding one means searching for it, with the O held over the bridge or along a path between the two hollows; module 09 computes such a path, and a saddle on it is its subject rather than this module's
- `np.linalg.norm(forces, axis=1)` is the length of each row of the forces array, so one force magnitude per atom, and `magnitudes.argmax()` is the position of the largest of them, which is how the cell names the atom carrying it
- In the printed output and the table:
    - The forces first, then the three modes in the same shape as the two sections above
    - `imaginary` is `True` on exactly one row, and that row's O moves almost entirely in the surface plane
    - The two real modes are higher than the two at the relaxed hollow site, one in the plane and one along the normal in both cases; that is a thing to notice and not a thing to report, because the forces printed above this table say the geometry is not a stationary point
    - `summary()` marks the imaginary mode with an `i` after both of its numbers, and prints a zero-point energy under the table as it did for the relaxed structure, which the next section is about

In [7]:
bridge = pt111_with_adsorbate("O", "bridge")
bridge.calc = calc
i_o_bridge = bridge.get_chemical_symbols().index("O")
bridge_forces = bridge.get_forces()
magnitudes = np.linalg.norm(bridge_forces, axis=1)

print("Largest force on any atom of the placed bridge structure (eV/Å)")
print(round(calc_fmax(bridge_forces), 4))
print("Symbol of the atom carrying it")
print(bridge[int(magnitudes.argmax())].symbol)
print("Force on the O along the surface normal (eV/Å)")
print(round(float(abs(bridge_forces[i_o_bridge, 2])), 4))
print("Force on the O in the surface plane (eV/Å)")
print(round(float(np.linalg.norm(bridge_forces[i_o_bridge, :2])), 4))

vib_bridge = Vibrations(bridge, indices=[i_o_bridge], name="work/vib_bridge")
vib_bridge.run()
frequencies_bridge = vib_bridge.get_frequencies()

print()
print("Modes")
print(len(frequencies_bridge))
print("Imaginary modes among them")
print(int((frequencies_bridge.imag != 0).sum()))

print()
print("The three modes of the O atom placed at the bridge site, not relaxed")
print(pd.DataFrame(mode_rows(vib_bridge, bridge, [i_o_bridge])).to_string(index=False))

buffer = io.StringIO()
vib_bridge.summary(log=buffer)
print()
print("What summary() writes, with an imaginary mode in the list")
print(buffer.getvalue())

Largest force on any atom of the placed bridge structure (eV/Å)
1.5143
Symbol of the atom carrying it
O
Force on the O along the surface normal (eV/Å)
1.514
Force on the O in the surface plane (eV/Å)
0.0318



Modes
3
Imaginary modes among them
1

The three modes of the O atom placed at the bridge site, not relaxed
 mode  frequency (cm⁻¹)  imaginary  O moves  O along z
    0             236.0       True     0.25      0.008
    1             334.9      False     0.25      0.000
    2             479.5      False     0.25      1.000

What summary() writes, with an imaginary mode in the list
---------------------
  #    meV     cm^-1
---------------------
  0   29.3i    236.0i
  1   41.5     334.9
  2   59.5     479.5
---------------------
Zero-point energy: 0.050 eV



### 6a. What a zero-point energy does with an imaginary mode

- `get_zero_point_energy()` returns a number for this structure too, and the number is wrong in a way nothing warns you about
    - It is half the sum of the real parts, so the imaginary mode enters it as zero: the value printed below is the two real modes and nothing else
    - The cell prints it next to half the sum of the real parts, to show that they are the same number, and next to the count of imaginary modes, which is the only thing in the cell that says the value is meaningless
- `HarmonicThermo` refuses the same list instead
    - With its default, `ignore_imag_modes=False`, it raises `ValueError: Imaginary vibrational energies are present.` before computing anything
    - `ignore_imag_modes=True` makes it drop every mode whose energy is not positive and warn that it did, so that warning is the trace of a decision you took rather than a fact about the structure
    - `warnings.catch_warnings(record=True)` collects the warnings raised inside the block into a list instead of letting them print, which is how the cell prints the text of that warning as a value
- The lesson is the pair of behaviours, not either one
    - Two tools reading the same list of modes: one hands back a plausible number in silence, the other stops
    - So the count of imaginary modes is checked and reported next to any zero-point energy, and a structure that has one goes back to be understood rather than into a free energy
- In the printed output:
    - The zero-point energy, half the sum of the real parts and the count of imaginary modes, the three numbers that belong together
    - The error `HarmonicThermo` raises, with its own text, and then the warning it gives when it is told to ignore the mode instead
    - `clean()` at the end removes this structure's displacement files, as the sections above did with theirs

In [8]:
bridge_energies = vib_bridge.get_energies()

print("Zero-point energy this structure reports (eV)")
print(round(vib_bridge.get_zero_point_energy(), 4))
print("Half the sum of the real parts of its mode energies (eV)")
print(round(0.5 * float(bridge_energies.real.sum()), 4))
print("Imaginary modes in the list it was computed from")
print(int((frequencies_bridge.imag != 0).sum()))

print()
print("What HarmonicThermo does with the same list by default")
try:
    HarmonicThermo(vib_energies=bridge_energies)
    print("no error raised")
except ValueError as error:
    print(type(error).__name__ + ": " + str(error))

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    HarmonicThermo(vib_energies=bridge_energies, ignore_imag_modes=True)

print("What it does when told to ignore them")
print([str(item.message) for item in caught])

print()
print("Displacement files removed by clean()")
print(vib_bridge.clean())

Zero-point energy this structure reports (eV)
0.0505
Half the sum of the real parts of its mode energies (eV)
0.0505
Imaginary modes in the list it was computed from
1

What HarmonicThermo does with the same list by default
ValueError: Imaginary vibrational energies are present.
What it does when told to ignore them
['1 imag modes removed']

Displacement files removed by clean()
7


### 6b. One model, two structures built the same way

- This is a Python error rather than a piece of chemistry, and it is in this notebook because it is easy to run into and hard to read: it appears when one model object is asked about a structure whose geometry is identical to the one it last calculated, and the two structures were built separately
    - The structures carry a key in `atoms.info` called `adsorbate_info`, and it is the surface builder underneath `fcc111` that puts it there, not the adsorbate: a bare `pt111_slab()` already carries it, holding the surface cell and the four site positions, before any adsorbate exists
    - So two separately built clean slabs run into this as readily as two structures with an adsorbate on them, and placing the adsorbate only adds one more entry to the same dictionary
    - Before it calculates, the MACE calculator compares the structure it is given with the one it last calculated; finding no difference in the geometry, it goes on to compare the two `atoms.info` dictionaries, and it skips a value only when that value is itself an array
    - `adsorbate_info` is not an array, it is a dictionary holding one, so it does get compared; comparing two dictionaries that hold two different arrays tries to turn a table of `True` and `False` into a single yes or no, and that raises `ValueError: The truth value of an array with more than one element is ambiguous`
- Which is why the error needs two structures built separately, and not just two calls
    - Ask the same structure twice, or ask a copy of it, and the two dictionaries hold the same objects; a comparison of two dictionaries that hold the same objects answers without comparing the arrays at all, and the cell shows that call going through
    - Build the structure a second time and the arrays are new objects with the same contents, which is the case that raises
    - The geometry has to match the one the model last calculated as well: any real difference is found first and the dictionaries are then never reached, which is why module 06's loop over four different sites never met this
- The fix is one line, `atoms.info.pop("adsorbate_info", None)`, on any structure the helper built
    - `pop(key, None)` removes a key from a dictionary, and the second argument is what it returns when the key was not there instead of raising, so the line is also safe on a structure read from a file, which carries no such key
    - Every structure the sections above hand to the model is either read back from a file, which carries no such key, or built by the helper exactly once, so none of them needed the line; your own loop over structures from the helper is where it will save you a confusing half hour
- `np.array_equal(a, b)` is `True` when two arrays have the same shape and the same values, used here to show that the two structures really are at the same geometry
- In the printed output:
    - The energy of the placed bridge structure, then the same call through a copy of it, which goes through and gives the same number
    - Then the same call on a second structure built by the helper, which raises, with the error's own text
    - Then that same call once more after the one-line fix, giving the energy the first call gave

In [9]:
print("Energy of the placed bridge structure (eV)")
print(round(float(bridge.get_potential_energy()), 4))

same = bridge.copy()
same.calc = calc

print()
print("A copy of it holds the same adsorbate_info object")
print(same.info["adsorbate_info"] is bridge.info["adsorbate_info"])
print("Energy through that copy (eV)")
print(round(float(same.get_potential_energy()), 4))

rebuilt = pt111_with_adsorbate("O", "bridge")
rebuilt.calc = calc

print()
print("A second structure from the helper is at the same geometry")
print(bool(np.array_equal(rebuilt.positions, bridge.positions)))
print("and holds an adsorbate_info object of its own")
print(rebuilt.info["adsorbate_info"] is bridge.info["adsorbate_info"])
print("What the model says when asked about it")
try:
    rebuilt.get_potential_energy()
    print("no error raised")
except ValueError as error:
    print(type(error).__name__ + ": " + str(error))

rebuilt.info.pop("adsorbate_info", None)

print()
print("Energy of that same structure after the one-line fix (eV)")
print(round(float(rebuilt.get_potential_energy()), 4))

Energy of the placed bridge structure (eV)


-157.8663

A copy of it holds the same adsorbate_info object
True
Energy through that copy (eV)
-157.8663



A second structure from the helper is at the same geometry
True
and holds an adsorbate_info object of its own
False
What the model says when asked about it
ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

Energy of that same structure after the one-line fix (eV)


-157.8663


## 7. The same analysis from the terminal: `mlip freq run`

- `mliprun` 0.5.0 added a command for what the sections above did by hand, and this section runs it on the two relaxed files and compares
    - Version 0.4.0, which earlier versions of this course were written against, had no such command; the cell prints the version of your environment, and `mlip --help` lists `freq` from 0.5.0 on
    - Underneath it is the same `Vibrations` of ASE, so it is the same calculation and should give the same numbers, and the table below checks that
- `mlip freq run --structure work/OH_fcc/opt_final.vasp --mlip mace --device cpu --indices 1,0 --output-dir work/freq_OH_fcc`
    - `--structure` is the relaxed file the Python sections read, and the command reads it the same way, constraint included
    - `--mlip mace` names the model, canon `C1`, as every command of this course does
    - `--indices` is the list of atoms to vibrate, as in `Vibrations`; leave it out and the command vibrates every atom that no `FixAtoms` holds, 19 atoms here and 115 force calls instead of 13
    - The O is typed first, `1,0`, the order that gave wrong numbers when it went to ASE unsorted, as the section on OH on the surface explained; the command sorts the list before it calls ASE, and the record keeps the list it actually used
    - `--output-dir` gives the run a folder of its own; without it the files land next to the structure, in `work/OH_fcc/`, and the frequency run's record would replace the record of the relaxation there, module 06's section on the single-point command
    - `delta` 0.01 Å and `nfree` 2 are the command's defaults, the same as ASE's, and the record writes them down with `"source": "default"`
- `run_freq` in the cell deletes the output folder before each run with `shutil.rmtree(out, ignore_errors=True)`, which is this command's version of `clean()`
    - The command keeps its displacements in a cache folder, `freq/`, as `Vibrations` did, and a second run reuses it
    - It also records what the cache was computed with, in `freq_cache.json`, and refuses to reuse a cache whose model, `delta` or starting forces differ; running this notebook again relaxes the structures again, the starting forces differ in their last digits, and without the deletion the command would stop with that refusal
    - Deleting the folder first is right here, because every run of the notebook starts from a new relaxation; on a long calculation that was interrupted, keeping the folder is what lets the command carry on where it stopped
- `signed(values)` turns ASE's complex frequencies into one signed number each, negative for an imaginary mode, which is how the command writes them, so the two can be compared entry by entry
- What the command adds over the Python of the sections above
    - Frequencies are signed: an imaginary mode is written as a negative number, so `-288.9` in the record is the `288.9` with `imaginary` true in the table of the section on OH on the surface
    - It counts the imaginary modes for you, `n_imaginary`, and counts a negative mode only when it is larger than `--imaginary-floor`, 10 cm⁻¹ by default, so numerical noise around zero is not called imaginary
    - It measures the largest force at the input geometry and compares it with the force target of the relaxation that produced the file, read from the run record next to that file in `work/OH_fcc/`; the next section is what happens when the two disagree
    - It writes a run record with the model, the versions and every setting: the list earlier versions of this course said you had to keep by hand
    - Two largest forces are reported: over the free atoms, the number a relaxation converges against, and over all atoms, which includes the force on the held bottom layer that `atoms.get_forces()` reports as zero, module 06's section on energy and forces
- In the printed output:
    - The version, then the two commands as you would type them, with the seconds each took
    - The table: for both structures the command and the Python agree on every frequency to the 0.1 cm⁻¹ printed and on the zero-point energy, and the OH row shows the list typed as `[1, 0]` and recorded as `[0, 1]`
    - Then the OH run's own numbers: the signed frequencies with the imaginary pair first, the largest force over the free atoms and over all atoms, 0.0452 against 0.1726 eV/Å, and the expected 0.05 eV/Å with the source `run_record`, which is the `--fmax` of the relaxation in that folder
    - The files it wrote: the cache folder `freq`, its `freq_cache.json`, the record, the table of frequencies `freq_frequencies.csv`, ASE's own `freq_summary.txt`, the full Hessian in `freq_vibrations.json`, and one trajectory per imaginary mode, `freq.0.traj` and `freq.1.traj`, which play the motion as a short film in `ase gui`


In [10]:
from importlib.metadata import version


def signed(values):
    """Complex frequencies from ASE as one signed number each, negative for an imaginary mode."""
    return [round(float(-v.imag if v.imag != 0 else v.real), 1) for v in values]


def run_freq(structure, atom_list, out, extra=()):
    """Run mlip freq run on one structure into its own folder, and return its run record."""
    shutil.rmtree(out, ignore_errors=True)
    result = run_mlip(["freq", "run",
                       "--structure", str(structure),
                       "--mlip", "mace",
                       "--device", "cpu",
                       "--indices", ",".join(str(i) for i in atom_list),
                       "--output-dir", str(out)] + list(extra))
    return json.loads((out / "mliprun_run.json").read_text()), result


print("mliprun version in this environment")
print(version("mliprun"))

rows = []
for label, folder, atom_list, python_values, python_zpe in (
        ("O at fcc", o_folder, [i_o], frequencies, zero_point),
        ("OH at fcc", oh_folder, [i_o_oh, i_h_oh], frequencies_oh, vib_oh.get_zero_point_energy())):
    print()
    record, _ = run_freq(folder / "opt_final.vasp", atom_list, WORK / ("freq_" + folder.name))
    results = record["stages"][-1]["results"]
    rows.append({"structure": label,
                 "indices typed": atom_list,
                 "indices recorded": record["parameters"]["indices"]["value"],
                 "imaginary modes": results["n_imaginary"],
                 "ZPE, command (eV)": round(results["zpe_eV"], 4),
                 "ZPE, Python (eV)": round(python_zpe, 4),
                 "same frequencies to 0.1 cm⁻¹": signed(python_values)
                 == [round(v, 1) for v in results["frequencies_cm-1"]]})
    last = (label, record, results)

print()
print("mlip freq run against the Python of the sections above, on the same relaxed files")
print(pd.DataFrame(rows).to_string(index=False))

label, record, results = last
print()
print("Frequencies the command recorded for " + label + ", signed (cm⁻¹)")
print([round(v, 1) for v in results["frequencies_cm-1"]])
print("Which of them it counts as imaginary")
print(results["imaginary_mask"])
print("Largest force at the input geometry, free atoms and all atoms (eV/Å)")
print([round(results["fmax_at_input_free_eV_per_A"], 4), round(results["fmax_at_input_all_eV_per_A"], 4)])
print("Force it expected, and where that expectation came from")
print([results["fmax_expectation"], results["fmax_expectation_source"]])
print("Model, and the mliprun version, as the record keeps them")
print([record["provenance"]["mlip_model"], record["provenance"]["mliprun_version"]])
print("Files the command wrote into " + str(WORK / ("freq_" + oh_folder.name)))
print(sorted(path.name for path in (WORK / ("freq_" + oh_folder.name)).iterdir()))


mliprun version in this environment


0.5.0

Command, as you would type it in the terminal
mlip freq run --structure work/O_fcc/opt_final.vasp --mlip mace --device cpu --indices 0 --output-dir work/freq_O_fcc



Seconds the command took
6.9

Command, as you would type it in the terminal
mlip freq run --structure work/OH_fcc/opt_final.vasp --mlip mace --device cpu --indices 1,0 --output-dir work/freq_OH_fcc



Seconds the command took
8.0

mlip freq run against the Python of the sections above, on the same relaxed files
structure indices typed indices recorded  imaginary modes  ZPE, command (eV)  ZPE, Python (eV)  same frequencies to 0.1 cm⁻¹
 O at fcc           [0]              [0]                0             0.0646            0.0646                          True
OH at fcc        [1, 0]           [0, 1]                2             0.2888            0.2888                          True

Frequencies the command recorded for OH at fcc, signed (cm⁻¹)
[-288.9, -288.9, 223.0, 338.2, 338.2, 3758.7]
Which of them it counts as imaginary
[True, True, False, False, False, False]
Largest force at the input geometry, free atoms and all atoms (eV/Å)
[0.0452, 0.1726]
Force it expected, and where that expectation came from
[0.05, 'run_record']
Model, and the mliprun version, as the record keeps them
['mace', '0.5.0']
Files the command wrote into work/freq_OH_fcc
['freq', 'freq.0.traj', 'freq.1.traj', 'f

### 7a. The stationary-point warning

- The section on an imaginary frequency ran an analysis on a placed, unrelaxed bridge structure on purpose, and had to read the forces itself to see that the geometry was not a stationary point; the command makes that check for you
- The structure is written to `work/O_bridge_placed/POSCAR`, with no relaxation, so there is no run record next to it to take an expected force from
    - With no record and no `--expect-fmax`, the command still measures the force and prints it, but has nothing to compare it with, and records the source of the expectation as `none`
    - `--expect-fmax 0.05` gives it the value to compare with, module 07's force target
- The warning never stops the run: the status is `completed`, the frequencies are written, and `fmax_warning` true in the record is what tells a later reader that they are curvatures at a point nothing sits at
- In the printed output:
    - 1.5143 eV/Å on the free atoms against the 0.05 expected, the source `explicit`, and `fmax_warning` true
    - The three frequencies of the section on an imaginary frequency, one of them negative, −236.0, which is the 236.0i of that section
    - The two warning lines the command printed: one for the force, and one for the imaginary mode and the zero-point energy that leaves it out


In [11]:
placed_folder = WORK / "O_bridge_placed"
placed_folder.mkdir(parents=True, exist_ok=True)
write(placed_folder / "POSCAR", pt111_with_adsorbate("O", "bridge"), format="vasp", direct=True, sort=True)
i_o_placed = read(placed_folder / "POSCAR").get_chemical_symbols().index("O")

record, result = run_freq(placed_folder / "POSCAR", [i_o_placed], WORK / "freq_O_bridge_placed",
                          extra=["--expect-fmax", "0.05"])
results = record["stages"][-1]["results"]

print()
print("Largest force on a free atom at the input geometry (eV/Å)")
print(round(results["fmax_at_input_free_eV_per_A"], 4))
print("Force it expected, and where that expectation came from")
print([results["fmax_expectation"], results["fmax_expectation_source"]])
print("fmax_warning in the record")
print(results["fmax_warning"])
print("Status of the run")
print(record["stages"][-1]["status"])
print("Frequencies it recorded, signed (cm⁻¹)")
print([round(v, 1) for v in results["frequencies_cm-1"]])
print("Imaginary modes it counts")
print(results["n_imaginary"])
print()
print("The warning lines the command printed")
print("\n".join(line for line in shorten(result.stdout).splitlines() if "⚠" in line))


Command, as you would type it in the terminal
mlip freq run --structure work/O_bridge_placed/POSCAR --mlip mace --device cpu --indices 0 --output-dir work/freq_O_bridge_placed --expect-fmax 0.05



Seconds the command took
7.5

Largest force on a free atom at the input geometry (eV/Å)
1.5143
Force it expected, and where that expectation came from
[0.05, 'explicit']
fmax_warning in the record
True
Status of the run
completed
Frequencies it recorded, signed (cm⁻¹)
[-236.0, 334.9, 479.5]
Imaginary modes it counts
1

The warning lines the command printed
⚠️  The free-component value is above the 0.050000 eV/Å expected from explicit. A geometry that is not a stationary point produces spurious imaginary modes; the run continued.
⚠️  1 imaginary mode(s) above 10 cm⁻¹, written as negative frequencies. ZPE above counts the real modes only.


## Check yourself

1. You relax an adsorbate to `--fmax 0.05`, the run reports `converged` true, and the frequency analysis of the
   adsorbate comes back with one mode at 112i cm⁻¹. What does that tell you, what does it not tell you, and what
   are the first two things you check?
2. You compute a zero-point energy for a structure whose lowest mode is imaginary and a perfectly ordinary number
   comes out. Why does it, and what does `HarmonicThermo` do when it is given the same list of modes?
3. A colleague sends you a zero-point energy of 0.065 eV for O on Pt(1 1 1) and asks you to put it into a free
   energy next to numbers of their own. What do you have to ask them before you can use it?

<details>
<summary>Answers</summary>

- It tells you that the geometry is not a minimum of that model: the energy falls away along that one direction,
  which is what a negative curvature means. It does not tell you that you have found a transition state; that needs
  a stationary point with exactly one imaginary mode, and a `converged` label at 0.05 eV/Å is not yet a stationary
  point for this purpose. Check the forces at that geometry first, the way the section on an imaginary frequency
  does, where a placed structure carrying 1.5143 eV/Å is shown not to be a stationary point at all. Then relax
  tighter and compute again, which is the section on the same OH relaxed tighter: there the target went from 0.05 to
  0.01 eV/Å, the run reached 0.0092 eV/Å, and the imaginary pair stayed, which is how that section knows the target
  was not the cause. Also check which atoms were in the Hessian, because that choice is part of the answer
- Because a zero-point energy is half the sum of the real parts of the mode energies, and an imaginary mode's energy
  has a real part of zero, so it contributes nothing and nothing is printed to say so; the section on what a
  zero-point energy does with an imaginary mode prints the two numbers side by side to show it. `HarmonicThermo`
  behaves the other way round: with its default it raises `ValueError: Imaginary vibrational energies are
  present.`, and it drops those modes only if you pass `ignore_imag_modes=True`, which makes it warn. One tool hands
  you a plausible number in silence and the other stops you, and the structure is not a minimum either way
- Which atoms were vibrated, because a zero-point energy over one O atom is that atom's three modes and not the
  structure's; the model and the head, canon `C1`, since their numbers have to come from the same one; the force
  target the geometry was relaxed to and its `converged` label, canon `K1`; `delta` and `nfree`, the displacement
  size and the number of displacements per direction, which are what the finite differences were taken with; and how
  many modes came out imaginary, because a zero-point energy hides those without a word. Then the same choices have
  to hold on both sides of the difference, which is the rule canon `C4` states for a force target. If they ran
  `mlip freq run`, the `mliprun_run.json` it wrote holds every one of these, so ask for that file

</details>